# 12주차 실습  서울 공공자전거에서 패턴과 관계 찾기

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

fonts = {font.name for font in fm.fontManager.ttflist}
for name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if name in fonts:
        plt.rcParams["font.family"] = name
        break
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 20)

In [ ]:
csv_path = Path("SeoulBikeData.csv")
if not csv_path.exists():
    csv_path = Path("../강의교재/data/SeoulBikeData.csv")
if not csv_path.exists():
    csv_path = Path("강의교재/data/SeoulBikeData.csv")
bike = pd.read_csv(csv_path, encoding="latin1")

In [ ]:
bike = bike.rename(columns={
    "Date": "date", "Rented Bike Count": "rented_count", "Hour": "hour",
    "Temperature(°C)": "temperature", "Humidity(%)": "humidity",
    "Wind speed (m/s)": "wind_speed", "Visibility (10m)": "visibility",
    "Dew point temperature(°C)": "dew_point", "Solar Radiation (MJ/m2)": "solar_radiation",
    "Rainfall(mm)": "rainfall", "Snowfall (cm)": "snowfall",
    "Seasons": "season", "Holiday": "holiday", "Functioning Day": "functioning_day"
})
bike["date"] = pd.to_datetime(bike["date"], format="%d/%m/%Y")
operating = bike[bike["functioning_day"] == "Yes"].copy()

In [ ]:
# 요일을 준비합니다. 날짜에서 추출한 숫자는 월요일 0 ~ 일요일 6입니다.
operating["weekday_number"] = operating["date"].dt.dayofweek
weekday_names = {0: "월", 1: "화", 2: "수", 3: "목", 4: "금", 5: "토", 6: "일"}
operating["weekday"] = operating["weekday_number"].map(weekday_names)
operating["is_rain"] = operating["rainfall"] > 0
weekday_order = ["월", "화", "수", "목", "금", "토", "일"]
print("운영한 시간의 기록:", len(operating), "행")
operating.head(3)

## 분석에 사용할 열

한 행은 **서울 전체의 한 날짜·한 시간에 대한 기록**입니다. 개인 한 명이나 대여소 하나의 기록이 아닙니다.

| 열 | 의미 |
|---|---|
| `date`, `hour` | 날짜와 시각(0~23시) |
| `rented_count` | 그 한 시간의 대여 건수 |
| `weekday_number`, `weekday` | 요일 숫자와 이름 |
| `temperature` | 기온(℃) |
| `season` | `Spring`, `Summer`, `Autumn`, `Winter` |
| `rainfall`, `is_rain` | 강수량(mm)과 비가 왔는지 여부 |
| `humidity`, `wind_speed`, `snowfall` | 습도, 풍속, 적설량 |
| `holiday` | 공휴일 여부(`Holiday`, `No Holiday`) |

`weekday_number < 5`는 월~금, `weekday_number >= 5`는 토·일입니다. **월~금에도 공휴일이 포함됩니다.** 공휴일 여부와 요일 구분을 혼동하지 않습니다.

문제의 평균 대여량은 **운영한 시간의 시간당 평균**입니다. `count()`는 유효한 시간 기록 수이며 사람 수나 하루 수가 아닙니다. 자료가 없는 비운영 시간을 0으로 채우지 않습니다.

## 문제 1 · 요일에 따라 대여량이 다른가

1. 요일별 평균 대여량과 유효한 시간 기록 수를 각각 구하세요.
2. 두 결과를 한 표로 합치고 월~일 순서로 정리하세요. 평균을 막대그래프로 그리세요.
3. 평균이 가장 높은 요일과 가장 낮은 요일을 코드로 찾으세요. 차이를 수치로 설명하세요.
4. 이 결과가 **하루 총대여량의 평균**과 다른 이유를 한 문장으로 쓰세요.

힌트: `groupby("weekday")["rented_count"].mean()`, `count()`, `pd.DataFrame()`, `.loc[weekday_order]`.
계산 표를 먼저 확인하고 다음 셀에서 그래프를 그리세요.

In [ ]:
# 문제 1 · 코드 작성 1


In [ ]:
# 문제 1 · 코드 작성 2


**분석 기록**

- 확인한 수치:
- 발견한 패턴:
- 추가로 확인할 내용:

## 문제 2 · 하루 안의 패턴을 월~금과 토·일로 나누기

1. `operating`에서 월~금은 `weekday`, 토·일은 `weekend`라는 표로 따로 추출하세요.
2. 각 표에서 시간대별 평균 대여량을 구하고 두 열의 비교표를 만드세요.
3. 두 집단의 선을 같은 그래프에 그리고 최고점 시각을 코드로 찾으세요.
4. 8시와 18시의 평균을 수치로 비교하세요. 아침과 저녁의 모양에 어떤 차이가 있습니까?
5. 평균을 계산한 시간 기록 수를 시간대별로 비교하세요. 이용 목적을 단정할 수 있는지도 쓰세요.

힌트: `operating[operating["weekday_number"] < 5]`, `groupby("hour")`, `idxmax()`, `.loc[[8, 18]]`.
두 표를 만드는 이름 `weekday`, `weekend`는 문제 3에서도 사용합니다.

In [ ]:
# 문제 2 · 코드 작성 1


In [ ]:
# 문제 2 · 코드 작성 2


In [ ]:
# 문제 2 · 코드 작성 3


**분석 기록**

- 확인한 수치:
- 발견한 패턴:
- 추가로 확인할 내용:

## 문제 3 · 다른 달에도 같은 시간 패턴이 나타나는가

문제 2의 **월~금 자료**에서 6월과 8월을 비교합니다.

1. 6월 자료를 `june`, 8월 자료를 `august`로 고르세요.
2. 각 달의 시간대별 평균을 한 그래프에 나타내세요. 시간대별 기록 수도 확인하세요.
3. 각 달의 최고점 시각을 코드로 찾고, 아침·저녁 모양의 공통점과 차이점을 쓰세요.
4. 두 달에서 비슷한 패턴이 나왔다면 어디까지 결론을 쓸 수 있습니까? 다르다면 어떤 조건을 더 확인하겠습니까?

날짜 조건 예시: `(weekday["date"] >= "2018-06-01") & (weekday["date"] < "2018-07-01")`.
8월은 8월 1일 이상, 9월 1일 미만으로 고릅니다.

In [ ]:
# 문제 3 · 코드 작성 1


In [ ]:
# 문제 3 · 코드 작성 2


In [ ]:
# 문제 3 · 코드 작성 3


**분석 기록**

- 확인한 수치:
- 발견한 패턴:
- 추가로 확인할 내용:

## 문제 4 · 기온과 대여량의 관계가 여름에도 같은가

전체 운영 자료에서는 기온과 대여량의 상관계수가 약 0.56이었습니다. 이 관찰을 출발점으로 **여름 안에서도 같은 관계가 뚜렷한지** 확인합니다.

1. `season == "Summer"`인 행을 `summer`로 추출하세요.
2. 전체와 여름의 기록 수, 상관계수, 기온 최솟값·최댓값을 각각 구하세요.
3. 여름의 기온–대여량 산점도를 그리고 전체 상관계수와 비교해 설명하세요.
4. 상관계수가 낮아졌을 때 “여름에는 기온과 아무 관계가 없다”고 결론내려도 됩니까?

힌트: `len()`, `corr()`, `min()`, `max()`, `plot(kind="scatter", x="temperature", y="rented_count", s=10, alpha=0.3)`.

In [ ]:
# 문제 4 · 코드 작성 1


In [ ]:
# 문제 4 · 코드 작성 2


**분석 기록**

- 확인한 수치:
- 발견한 패턴:
- 추가로 확인할 내용:

## 문제 5 · 같은 계절과 같은 시각으로 더 나누기

여름에는 아침·저녁과 낮의 기온, 대여량이 함께 다를 수 있습니다.

1. `summer`에서 8시 자료 `summer8`, 18시 자료 `summer18`을 고르세요.
2. 각 집단의 기록 수와 기온–대여량 상관계수를 구하세요.
3. 각각 산점도를 그리세요. 두 그래프의 축 범위는 기온 15~40℃, 대여량 0~4000건으로 맞추세요.
4. 전체 → 여름 → 여름의 같은 시각으로 비교 범위를 좁히면서 설명을 어떻게 바꾸어야 하는지 쓰세요.

힌트: 조건 필터링, `corr()`, `plt.xlim(15, 40)`, `plt.ylim(0, 4000)`.
여름의 8시와 18시 사이에서 상관계수의 대소만으로 인과 효과를 비교하지 않습니다.

In [ ]:
# 문제 5 · 코드 작성 1


In [ ]:
# 문제 5 · 코드 작성 2


In [ ]:
# 문제 5 · 코드 작성 3


**분석 기록**

- 확인한 수치:
- 발견한 패턴:
- 추가로 확인할 내용:

## 문제 6 · 평균 대신 대여가 많은 시간의 비율 보기

한 시간의 대여량이 **1500건 이상**이면 `high_demand`를 `True`로 분류합니다. 1500건은 이 실습의 비교 기준입니다. 실제 자전거 부족 여부나 대여소 혼잡을 뜻하지 않습니다.

1. `operating`의 복사본을 `demand`로 만들고 `high_demand` 열을 추가하세요.
2. 강수 여부별 전체 시간 기록 수, 1500건 이상인 기록 수, 그 비율(%)을 각각 구하세요.
3. 계산 결과를 한 표로 합치고 비율을 막대그래프로 그리세요.
4. 두 비율의 차이를 **%p**로 쓰고, 분모가 무엇인지 설명하세요.

힌트: `copy()`, `>= 1500`, `groupby("is_rain")`, `count()`, `sum()`.
비율 = 해당 집단의 1500건 이상 시간 수 ÷ 해당 집단의 전체 유효 시간 수 × 100.

In [ ]:
# 문제 6 · 코드 작성 1


In [ ]:
# 문제 6 · 코드 작성 2


**분석 기록**

- 확인한 수치:
- 발견한 패턴:
- 추가로 확인할 내용:

## 문제 7 · 여름 18시에서도 비율 차이가 유지되는가

1. 문제 6의 `demand`에서 **여름이면서 18시**인 기록을 `demand18`로 고르세요.
2. 같은 방식으로 강수 여부별 시간 수, 1500건 이상 시간 수, 비율을 구하세요.
3. 문제 6의 전체 결과와 비교하세요. 비율의 수준과 차이의 방향은 어떻게 달라집니까?
4. 비가 온 집단의 표본 크기와 그 비율을 계산한 분자·분모를 직접 쓰세요. 비율만 보고 설명할 때 어떤 점을 조심해야 합니까?

힌트: `(demand["season"] == "Summer") & (demand["hour"] == 18)`.

In [ ]:
# 문제 7 · 코드 작성 1


**분석 기록**

- 확인한 수치:
- 발견한 패턴:
- 추가로 확인할 내용:

## 문제 8 · 따뜻한데 대여량이 적은 기록 추적하기

1. `operating`에서 **기온 20℃ 이상이면서 대여량 200건 미만**인 기록을 `unusual`로 추출하세요.
2. 대여량이 작은 순으로 정렬해 앞의 5개를 확인하세요. 대여량이 같으면 날짜, 시각 순서로 정렬합니다. 날짜, 시각, 기온, 대여량, 강수량, 습도, 계절을 표시하세요.
3. 첫 기록을 `chosen`으로 고르고, 같은 날짜의 운영 기록을 원자료에서 찾아 시간순으로 확인하세요.
4. 선택한 기록과 그날의 다른 시간에 나타난 단서를 바탕으로 **확인한 사실 두 가지, 가능한 설명 한 가지, 추가로 필요한 기록 한 가지**를 쓰세요.

힌트: 조건 두 개를 `&`로 연결하기, `sort_values(["rented_count", "date", "hour"])`, `iloc[0]`, `chosen["date"]`.
이 조건은 탐색을 위한 기준입니다. 조건에 해당한다고 오류로 판단하거나 삭제하지 않습니다.

In [ ]:
# 문제 8 · 코드 작성 1


In [ ]:
# 문제 8 · 코드 작성 2


**분석 기록**

- 확인한 수치:
- 발견한 패턴:
- 추가로 확인할 내용:

## 문제 9 · 분류 기준을 바꾸어도 같은 설명이 가능한가

문제 7의 **여름 18시 기록**을 그대로 사용하되 대여가 많다고 분류하는 기준을 바꿉니다.

1. `demand18`의 복사본을 만들어 1200건 이상과 1800건 이상을 나타내는 열을 각각 추가하세요.
2. 각 기준에서 강수 여부별 해당 건수와 비율을 따로 구하세요.
3. 1200·1500·1800건 기준의 비율을 한 표로 합치세요. 기준을 바꾸어도 비교 방향이 같은지 설명하세요.
4. 비가 온 집단의 기록 수를 근거로, 한 가지 기준에서 나온 비율을 얼마나 안정적으로 해석할 수 있는지 쓰세요.

힌트: `>= 1200`, `>= 1800`, 불리언 열의 `sum()`, 문제 7에서 계산한 `rain18_count`.
반복문 없이 기준별로 같은 계산을 한 번씩 작성해도 됩니다.

In [ ]:
# 문제 9 · 코드 작성 1


In [ ]:
# 문제 9 · 코드 작성 2


**분석 기록**

- 확인한 수치:
- 발견한 패턴:
- 추가로 확인할 내용:

## 문제 10 · 남은 의문을 직접 확인하기

문제 1~9의 결과에서 남은 의문을 하나 골라 **추가 분석 질문**을 작성하고 코드로 답하세요. 이미 계산한 결과를 다시 나열하는 질문은 피합니다.

다음은 질문의 방향 예시입니다. 예시 중 하나를 구체화하거나 다른 질문을 작성해도 됩니다.

- 여름 18시에서 비가 오지 않은 기록만 골라도 기온–대여량 관계가 약한가?
- 월~금과 토·일의 아침·저녁 차이는 겨울에도 같은가?
- 따뜻하지만 대여량이 적은 후보는 특정 시각이나 강수 조건에 몰려 있는가?

**작성 항목:** 질문, 필요한 열, 비교 조건, 관측 단위, 기록 수, 결과 표 또는 그래프, 근거 수치, 처음 생각에서 수정한 부분.

In [ ]:
# 문제 10 · 코드 작성 1


In [ ]:
# 문제 10 · 코드 작성 2
